In [2]:
from pathlib import Path

import pandas as pd
import surprise

from tarea_01.utils import calculate_metrics


In [3]:
dataset_path = Path('../dataset')
train_df = pd.read_csv(dataset_path / 'train.csv')
train_df.date = pd.to_datetime(train_df.date)

test_df = pd.read_csv(dataset_path / 'validation.csv')
test_df.date = pd.to_datetime(test_df.date)

reader = surprise.reader.Reader(rating_scale=(0,5))

train_data = surprise.Dataset.load_from_df(
    train_df[["user_id", "recipe_id", "rating"]], reader=reader
)
trainset = train_data.build_full_trainset()

test_data = surprise.Dataset.load_from_df(
    test_df[["user_id", "recipe_id", "rating"]], reader=reader
)
testset = test_data.construct_testset(
    test_data.raw_ratings
)

metrics = {}


In [4]:
from surprise import AlgoBase, PredictionImpossible


class BestRated(AlgoBase):

    def __init__(self):
        AlgoBase.__init__(self)

    def fit(self, trainset):
        AlgoBase.fit(self, trainset)
        # Compute the mean rating for each item
        self.item_means = {}
        for i, ratings in trainset.ir.items():
            self.item_means[i] = sum(r for _, r in ratings) / len(ratings)
        return self

    def estimate(self, u, i):
        if not self.trainset.knows_item(i):
            raise PredictionImpossible("Item is unknown.")
        return self.item_means[i]


class MostPopular(AlgoBase):

    def __init__(self):
        AlgoBase.__init__(self)

    def fit(self, trainset):
        AlgoBase.fit(self, trainset)
        # Count number of interactions per item
        self.item_counts = {i: len(ratings) for i, ratings in trainset.ir.items()}
        return self

    def estimate(self, u, i):
        if not self.trainset.knows_item(i):
            raise PredictionImpossible("Item is unknown.")
        # Returns number of interactions as the score
        return self.item_counts[i]

In [5]:
# Baseline ALS

model = surprise.BaselineOnly()
model.fit(trainset)

metrics['ALS'] = calculate_metrics(
    model=model,
    testset=testset,
    k=10,
    threshold=4.0,
)

Estimating biases using als...


In [ ]:
# Baseline Random

import random

# 1. Obtener el catálogo completo y los ítems vistos por cada usuario en train
all_items = list(train_df["recipe_id"].unique())
train_user_items = train_df.groupby("user_id")["recipe_id"].apply(set).to_dict()

# El testset original de Surprise solo tiene los casos reales
real_testset = test_data.construct_testset(test_data.raw_ratings)
test_users = set([uid for uid, _, _ in real_testset])

evaluation_testset = []

# 2. Agregar los casos reales de validación (Ground Truth)
for uid, iid, r_ui in real_testset:
    evaluation_testset.append((uid, iid, r_ui))

# 3. Inyectar N ítems aleatorios (negativos) por cada usuario en test
num_negatives = 100 

for uid in test_users:
    # Ítems que el usuario ya vio en train para no sugerirlos como negativos
    interacted_items = train_user_items.get(uid, set())
    
    negatives_added = 0
    while negatives_added < num_negatives:
        random_item = random.choice(all_items)
        # Asegurar que el ítem no esté en el train del usuario
        if random_item not in interacted_items:
            # Asignamos un rating de 0.0 porque asumimos que no le interesa (negativo)
            evaluation_testset.append((uid, random_item, 0.0))
            negatives_added += 1

# 4. Entrenar y evaluar
model = surprise.prediction_algorithms.random_pred.NormalPredictor()
model.fit(trainset)

# Usar el nuevo evaluation_testset
metrics['Random'] = calculate_metrics(
    model=model,
    testset=evaluation_testset,  # <-- Usamos el testset con negativos
    k=10,
    threshold=4.0,
)

In [7]:
model = BestRated()
model.fit(trainset)

metrics['Best rated'] = calculate_metrics(
    model=model,
    testset=testset,
    k=10,
    threshold=4.0,
)

In [8]:
model = MostPopular()
model.fit(trainset)

metrics["Most popular"] = calculate_metrics(
    model=model,
    testset=testset,
    k=10,
    threshold=4.0,
)

In [9]:
pd.DataFrame(metrics).T

,RMSE,Recall@10,Precision@10,NDCG@10,MAP@10,Diversity,Novelty
ALS,0.8818,0.9464,0.9410,0.9767,0.9625,0.8418,11.8891
Random,1.1050,0.9458,0.9399,0.9713,0.9545,0.8411,11.8995
Best rated,0.9210,0.9463,0.9409,0.9759,0.9615,0.8400,11.9198
Most popular,0.9860,0.9459,0.9401,0.9727,0.9567,0.8409,11.9008
